Import des Bibliothèques 

In [33]:
# Ajouter les fonction de data cleaning dans cette feuille 

#Bibliothèques 
import pandas as pd 
import numpy as np 
from io import StringIO

print ("librairies importées")


librairies importées


Code Ana 

day ahead prices 

In [ ]:


def harmonize_data_DayAhead_price(chemin_fichier):
    print("Étape 1 : Lecture du fichier CSV")
    try:
        with open(chemin_fichier, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()

        # Retire les guillemets externes et normalise les doubles guillemets
        cleaned_lines = []
        for line in raw_lines:
            line = line.strip()
            if line.startswith('"') and line.endswith('"'):
                line = line[1:-1]  # retire uniquement le premier et dernier guillemet
            line = line.replace('""', '"')
            cleaned_lines.append(line)

        cleaned_content = "\n".join(cleaned_lines)
        df_tot = pd.read_csv(StringIO(cleaned_content), sep=',', quotechar='"')
        df_tot.columns = df_tot.columns.str.strip()
        print("Lecture réussie. Dimensions :", df_tot.shape)
        print("Colonnes :", df_tot.columns.tolist())
    except Exception as e:
        print("Erreur lors de la lecture du fichier :", e)
        return None

    print("\nÉtape 2 : Sélection des colonnes utiles")
    try:
        df = df_tot[["MTU (UTC)", "Day-ahead Price (EUR/MWh)"]].copy()
        print("Colonnes sélectionnées. Aperçu :")
        print(df.head())
    except KeyError as e:
        print("Erreur : colonnes manquantes", e)
        return None

    print("\nÉtape 3 : Séparation de la plage horaire en Date_start et Date_end")
    try:
        df[["Date_start", "Date_end"]] = df["MTU (UTC)"].str.split(" - ", expand=True)
        print("Séparation réussie. Aperçu :")
        print(df.head())
    except Exception as e:
        print("Erreur lors de la séparation :", e)
        return None

    print("\nÉtape 4 : Suppression de la colonne originale MTU (UTC)")
    df = df.drop(columns=["MTU (UTC)"])
    print("Colonnes restantes :", df.columns.tolist())

    print("\nÉtape 5 : Conversion des dates en datetime")
    try:
        df["Date_start"] = pd.to_datetime(df["Date_start"], format="%d/%m/%Y %H:%M:%S", errors='coerce')
        df["Date_end"] = pd.to_datetime(df["Date_end"], format="%d/%m/%Y %H:%M:%S", errors='coerce')
        print("Conversion réussie. Types :")
        print(df.dtypes)
        print("Aperçu des premières lignes :")
        print(df.head())
    except Exception as e:
        print("Erreur lors de la conversion :", e)
        return None

    print("\nÉtape 6 : Conversion du prix en float")
    try:
        df["Day-ahead Price (EUR/MWh)"] = pd.to_numeric(df["Day-ahead Price (EUR/MWh)"], errors='coerce')
        print("Conversion réussie. Aperçu :")
        print(df.head())
    except Exception as e:
        print("Erreur lors de la conversion du prix :", e)
        return None

    print("\nToutes les étapes ont été effectuées avec succès.")
    return df

# Test de la fonction
path = "./Data/Day-ahead Prices_2024-2025.csv"
df_clean = harmonize_data_DayAhead_price(path)

Étape 1 : Lecture du fichier CSV
Lecture réussie. Dimensions : (8783, 10)
Colonnes : ['MTU (UTC)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)', 'Intraday Period (UTC)', 'Intraday Price (EUR/MWh)\n01/01/2024 00:00:00 - 01/01/2024 01:00:00,BZN|FR"', 'Without Sequence', '0.01', 'Unnamed: 8', 'Unnamed: 9']

Étape 2 : Sélection des colonnes utiles
Colonnes sélectionnées. Aperçu :
                                   MTU (UTC)  Day-ahead Price (EUR/MWh)
0  01/01/2024 01:00:00 - 01/01/2024 02:00:00                       0.00
1  01/01/2024 02:00:00 - 01/01/2024 03:00:00                      -0.01
2  01/01/2024 03:00:00 - 01/01/2024 04:00:00                      -0.03
3  01/01/2024 04:00:00 - 01/01/2024 05:00:00                      -0.02
4  01/01/2024 05:00:00 - 01/01/2024 06:00:00                      -0.04

Étape 3 : Séparation de la plage horaire en Date_start et Date_end
Séparation réussie. Aperçu :
                                   MTU (UTC)  Day-ahead Price (EUR/MWh)  \
0  01/01/2024 

In [4]:

# Marie : Prix de l'énergie sur le marché SPOT
def clean_data_price (chemin_fichier):
    df = pd.read_csv(chemin_fichier)
    if df["BZN|FR"].isna().all(): # on vérifie que la colonne est vide avant de la supprimer
        df = df.drop(columns=["BZN|FR"])
        print('La colonne "BZN|FR" était vide et a été supprimée.')
    else:
        print('La colonne "BZN|FR" contient des données.')
    df=df.drop(columns=["Currency"])
    df[["Date_start", "Date_end"]] = df["MTU (CET/CEST)"].str.split(" - ", expand=True) 
    df = df.drop(columns=["MTU (CET/CEST)"]) # on supprime l'ancienne colonne de la plage horaire
    df["Date_start"] = pd.to_datetime(df["Date_start"], format="%d.%m.%Y %H:%M", utc = True).dt.tz_localize(None) # on change de le format de la date de début et de fin
    df["Date_end"] = pd.to_datetime(df["Date_end"], format="%d.%m.%Y %H:%M", utc = True).dt.tz_localize(None) # on change de le format de la date de fin
    print(df.shape)
    print(df.head())
    return df
#clean_data_price(".\Data\Day-ahead Prices_2019-2020.csv")



In [ ]:
# Marine : Température & pseudo rayonnement
def clean_data_temperature (chemin_fichier): 
    fichier = pd.read_csv(chemin_fichier, sep=";")
    fichier["Horodate"] = pd.to_datetime(fichier["Horodate"], utc = True)
    fichier = fichier.drop(columns=["Année-Mois-Jour"])
    fichier = fichier.drop(columns = ["Année"])
    fichier = fichier.drop(columns = ["Mois"])
    fichier = fichier.drop(columns = ["Jour"])
    print(fichier.isna().sum())
    print(fichier.isnull().sum() )
    print(fichier.duplicated().sum())
    print(fichier.info())
    print(fichier.head())
    print(fichier.describe()) 
    return fichier

  